# ShopDesk, Module 2 Section 1 Lab 2: Scoped Tools and tool_choice

A beginner-friendly notebook on **who gets which tools** and **how hard the model is made to
use them**. We give different ShopDesk agents **full versus scoped** tool sets, then use the
`tool_choice` parameter (`auto`, `any`, and a forced tool) to control tool use, including
forcing a tool to guarantee **structured output**. Built on the **base Anthropic SDK**,
running **Sonnet** (`claude-sonnet-4-6`) through your **Anthropic API key**.

## The real-world scenario

Handing every agent every tool sounds convenient, but it hurts: more tools means more chances
to pick the wrong one, and a shipping agent that *can* issue refunds is a risk you did not need
to take. Scoping tools per role fixes both. Separately, some steps must produce a machine-readable
result every time, and `tool_choice` is how you guarantee that.

The question this lab answers: **how do you give each agent only the tools its role needs, and
how do you force a tool call when you need deterministic, structured output?**

## Objectives

- Assign **full vs scoped** tool sets to different agents and see why scoping helps.
- Use `tool_choice`: **auto** (model decides), **any** (must use some tool), and a **forced**
  tool (must use that exact tool).
- Force a tool to **guarantee structured output** for a deterministic workflow step.

## What you'll observe

- A scoped shipping agent literally cannot issue a refund: the tool is not in its set.
- With `auto`, a chit-chat message gets a text reply and no tool; with a forced tool, even that
  message produces the structured tool call.
- Forcing the output tool yields a guaranteed JSON payload every time.

## How to run

Run top to bottom. The tool sets and the output schema are pure Python and run anywhere. The
`tool_choice` cells call Claude, so paste a real key into **Setup 2/3** and re-run from the top;
otherwise they skip. `tool_choice` is a base Messages API parameter, so this lab uses the base
SDK.

## 0. Setup

**This cell:** installs the packages. This lab uses only the **base Anthropic SDK**,
because `tool_choice` is a parameter of the Messages API. (The Agent SDK does not expose
`tool_choice`; it controls tools with `allowed_tools` instead.)

In [ ]:
# ===== SETUP 1/3 - install the base SDK =====
%pip install -q anthropic python-dotenv

**This cell:** imports, the model, and the `RUN_LIVE` switch so live calls fire only with a
real key.

In [ ]:
# ===== SETUP 2/3 - imports, the model, and a live/offline switch =====
import os                                       # read the API key from the environment
import json                                     # print structured tool inputs
import anthropic                                # the base Anthropic SDK (synchronous)

try:                                            # load a .env file if present
    from dotenv import load_dotenv              #   import the loader
    load_dotenv()                               #   read .env into environment variables
except Exception:                               # not installed? that is fine
    pass                                        #   set the key another way

MODEL = "claude-sonnet-4-6"                      # the Sonnet model every call will use

os.environ.setdefault("ANTHROPIC_API_KEY", "sk-ant-...")     # placeholder unless you set a real key
_key = os.environ["ANTHROPIC_API_KEY"]           # read whatever key is set
RUN_LIVE = _key.startswith("sk-ant-") and _key != "sk-ant-..."   # True only for a real key
print("live model calls:", "ON" if RUN_LIVE else "OFF (using a placeholder key)")

**This cell:** the shared input schema and a small `run()` helper that makes one call with
a given tool set and `tool_choice`, and reports whether the model produced a tool call or text.
Every experiment below uses it.

In [ ]:
# ===== SETUP 3/3 - the shared schema and a one-call helper =====
ARG = {"type": "object", "properties": {"order_id": {"type": "string"}}, "required": ["order_id"]}

def run(query, tools, choice):                     # (query, tools, tool_choice) -> what happened
    client = anthropic.Anthropic()                 #   the LLM client (reads the key)
    r = client.messages.create(                    #   one Messages API call
        model=MODEL, max_tokens=300, tools=tools,   #   the available tools
        tool_choice=choice,                        #   auto / any / forced
        messages=[{"role": "user", "content": query}])
    for b in r.content:                            #   look for a tool call first
        if b.type == "tool_use":
            return ("tool", b.name, b.input)       #     the tool and its structured input
    text = "".join(b.text for b in r.content if b.type == "text")   # otherwise plain text
    return ("text", text, None)
print("run() helper ready")

### Two separate controls

**Which tools an agent has** and **how hard it is made to use them** are different levers:

- **Scoped tool sets** decide *availability*: a tool the agent does not have cannot be chosen,
  which both improves selection accuracy (fewer distractors) and removes whole classes of
  mistakes (a shipping agent cannot refund).
- **`tool_choice`** decides *obligation*: `auto` lets the model answer with text, `any` forces
  some tool, and a forced tool forces that exact one. Forcing is how you guarantee structured
  output for a deterministic step.

---

### 🎯 Lab objective - scope the tools, then control the choice

**What you build:** a full tool set and role-scoped subsets, plus experiments across the three
`tool_choice` modes, ending with a forced output tool.

**Why it helps you build real solutions:** scoping is your first line of both accuracy and safety,
and `tool_choice` is how you turn a probabilistic call into a guaranteed one when a step must not
fail.

**How you'll see it:** the scoped agent's options exclude refunds, and the forced tool returns a
structured payload every time.

**This cell:** the **full** tool set: everything ShopDesk can do. Handing all of this to
every agent is the tempting but risky default we will scope down next.

In [ ]:
# ===== the full tool set (everything) =====
check_status  = {"name": "check_status",  "description": "Return the delivery status of an order.", "input_schema": ARG}
track_package = {"name": "track_package", "description": "Return the carrier tracking link for a shipped order.", "input_schema": ARG}
order_details = {"name": "order_details", "description": "Return the items and total price of an order.", "input_schema": ARG}
refund_lookup = {"name": "refund_lookup", "description": "Return whether an order is refundable and issue the refund.", "input_schema": ARG}

FULL = [check_status, track_package, order_details, refund_lookup]   # every tool
print("FULL tools:", [t["name"] for t in FULL])

**This cell:** the **scoped** subsets, one per role. The shipping agent gets only shipping
tools; the refund agent gets only the refund tool. Notice the shipping set has no `refund_lookup`
at all, so that agent cannot issue a refund even if a request tries to make it.

In [ ]:
# ===== scoped tool sets, one per role =====
SHIPPING_SCOPED = [check_status, track_package]     # a shipping agent's tools only
REFUND_SCOPED   = [refund_lookup]                   # a refund agent's tools only

print("shipping agent can use:", [t["name"] for t in SHIPPING_SCOPED])
print("refund agent can use:  ", [t["name"] for t in REFUND_SCOPED])
print("shipping agent can refund?", any(t["name"] == "refund_lookup" for t in SHIPPING_SCOPED))

**This cell:** shows scoping as a **guarantee**. We send a refund-flavoured request to the
scoped shipping agent with `tool_choice={"type":"any"}`. Because `refund_lookup` is not in its
set, the agent is structurally unable to refund; it can only reach for a shipping tool.

In [ ]:
# ===== a scoped agent cannot step outside its role =====
sneaky = "Refund order A2 and also tell me where it is."   # tries to pull in a refund
if RUN_LIVE:                                        # needs a real key
    kind, name, _ = run(sneaky, SHIPPING_SCOPED, {"type": "any"})   # forced to pick from shipping tools
    print("shipping-scoped picked:", name, "(refund_lookup is not even available)")
    kind2, name2, _ = run(sneaky, FULL, {"type": "any"})           # with FULL, refund becomes reachable
    print("full-access picked:   ", name2, "(refund IS available here, which is the risk)")
else:
    print("[skipped] expected: the scoped agent can only pick check_status or track_package;")
    print("          with FULL access it could pick refund_lookup. Scoping removes that path.")

**This cell:** `tool_choice` mode one, **auto** (the default). The model may answer with
text instead of a tool. We send a chit-chat message: with `auto`, ShopDesk replies in words and
calls no tool, which is the right behaviour for a greeting.

In [ ]:
# ===== auto: the model may choose NOT to use a tool =====
if RUN_LIVE:                                        # needs a real key
    kind, payload, _ = run("Hi there, thanks for the help earlier!", FULL, {"type": "auto"})
    print("auto on a greeting ->", kind, "|", (payload if kind == "text" else payload))
else:
    print("[skipped] expected: 'text' with a friendly reply, and NO tool call.")

**This cell:** mode two, **any**. The model must use one of the tools, even on that same
greeting. This is useful when every turn must be tool-mediated, though it can force a tool onto an
input that did not really need one.

In [ ]:
# ===== any: the model MUST use some tool =====
if RUN_LIVE:                                        # needs a real key
    kind, name, _ = run("Hi there, thanks for the help earlier!", FULL, {"type": "any"})
    print("any on a greeting ->", kind, "| tool:", name, "(forced to pick something)")
else:
    print("[skipped] expected: 'tool' - 'any' forces a tool even when a greeting did not need one.")

**This cell:** an **output tool** whose input schema is the structured result we want. This
is the trick behind guaranteed structured output: define the shape you need as a tool, then force
that tool. Its `input` becomes your JSON payload.

In [ ]:
# ===== an output tool: its input_schema is the shape we want back =====
record_triage = {                                   # forcing this tool guarantees this structure
    "name": "record_triage",
    "description": "Record the triage result for a ticket.",
    "input_schema": {
        "type": "object",
        "properties": {
            "order_id": {"type": "string"},                                   # which order
            "intent":   {"type": "string", "enum": ["shipping", "refund", "other"]},   # what it is
            "priority": {"type": "string", "enum": ["low", "medium", "high"]},          # how urgent
        },
        "required": ["order_id", "intent", "priority"],
    },
}
print("output tool ready:", record_triage["name"], list(record_triage["input_schema"]["properties"]))

**This cell:** mode three, a **forced tool**: `tool_choice={"type":"tool","name":
"record_triage"}`. The model must call exactly that tool, so its `input` comes back as a
structured payload matching the schema, every time, with no prose to parse. This is the
deterministic structured-output pattern.

In [ ]:
# ===== forced tool: guaranteed structured output =====
FORCE = {"type": "tool", "name": "record_triage"}   # force this exact tool
ticket = "Customer says A2 is past the window but wants their money back, and is upset."
if RUN_LIVE:                                        # needs a real key
    kind, name, data = run(ticket, [record_triage], FORCE)   # must call record_triage
    print("forced ->", kind, "| tool:", name)
    print("structured payload:", json.dumps(data))  #   a guaranteed dict matching the schema
else:
    print("[skipped] expected: a tool call to record_triage with, e.g.,")
    print('          {"order_id":"A2","intent":"refund","priority":"high"} - guaranteed structure.')

**This cell:** the connection to the **Agent SDK**. Scoping is productised there as
`AgentDefinition(tools=[...])` per subagent (as in Module 1's coordinator lab). `tool_choice`,
however, is not exposed by the Agent SDK, so when you need forced/structured output you reach for
the base Messages API, exactly as above.

In [ ]:
# ===== how these map to the Agent SDK =====
print("scoping  -> AgentDefinition(tools=[...])   # each subagent sees only its role's tools")
print("forcing  -> base Messages API tool_choice  # Agent SDK does not expose tool_choice")

| anti-pattern | what to do instead |
|---|---|
| give every agent every tool | scope each agent to its role's tools |
| rely on a prompt to keep an agent in its lane | remove the out-of-role tool from its set entirely |
| parse JSON out of a free-text reply | force an output tool so the payload is structured |
| use `any` everywhere | use `auto` when text is fine; force only when a step must produce structure |

**Lesson:** control tools with two separate levers. **Scoping** decides what an agent can do
at all, improving accuracy and closing off whole classes of mistakes. **`tool_choice`** decides
how hard it must use them: `auto` for flexibility, `any` to require a tool, and a forced tool to
guarantee a specific, structured result for a deterministic step.

---

## Recap - scope and choice

| Lever | Setting | Effect |
|---|---|---|
| Scoping | full vs role-scoped `tools` | fewer distractors, and out-of-role actions become impossible |
| tool_choice | `{"type":"auto"}` | model may answer with text |
| tool_choice | `{"type":"any"}` | model must use some tool |
| tool_choice | `{"type":"tool","name":...}` | model must use that exact tool (structured output) |

One principle to carry forward: **give each agent only the tools its role needs, and force a tool
only when the step must produce structure.** To run live, paste a real key into **Setup 2/3** and
re-run from the top. Then try it: force `record_triage` on a plain greeting and watch it still
return a valid structured payload.